# Step 12b — E5-large-v2 ablation: MAX_LEN 256 -> 384 (single-knob audit)

**Why this exists:** post-comparison with a friend's SPECTER2 fine-tune that scored Public LB **0.71254** (vs ours 0.69972, delta +0.013), we audited config bias. Their winning recipe used `MAX_LEN=384` with the same `abstracts_merged_v2.csv` cache we use. After our v3+max_len=384+unconstrained-tuner attempt regressed to 0.68718, we froze `MAX_LEN=256` for ALL subsequent fine-tunes (SciNCL/SciBERT/BGE/E5) without re-isolating the variable. That's the L6 trap turned upside down — using "never change two knobs at once" to rationalise never changing the knob at all.

**Hypothesis:** the regression at 0.68718 was caused by `v3 cache + max_len 384 + batch=12 + constrained tuner` together (multi-knob). With v2 cache + max_len=384 + the proven step12 recipe, OOF should improve because:
- E5/BGE-large hidden=1024 (vs SPECTER2 768) — more capacity to use longer context
- Long abstracts (>256 tokens) tend to be full conference papers (label 4-5); short ones are proceedings (label 1) — truncating at 256 collapses discriminative signal
- v2 cache content quality is the same regardless of max_len; only context length changes

**ONE-knob change vs step12:** `MAX_LEN: 256 -> 384`. Nothing else moves.

**Decision after fold 1:**
- best per-fold ≥ step12 fold-1 (~0.66) → continue
- best per-fold < 0.60 → abort, max_len=384 hurts E5 (different from SPECTER2)

**Decision after full 15-fold (per L11/L13):**
- OOF QWK > 0.6496 (step12) by ≥ 0.005 → bias confirmed; promote as new anchor
- OOF QWK within ±0.005 → max_len doesn't matter for E5; drop
- OOF QWK drops ≥ 0.005 → E5 different from SPECTER2 here; drop and codify lesson

**Recipe (mirrors step 12 — only MAX_LEN differs):**
- Model: `intfloat/e5-large-v2` loaded with `torch_dtype=fp32`, bf16 autocast at train
- Input: `"passage: " + title [SEP] abstract`, **max_len 384** (was 256)
- Pooling: mean pool with attention mask
- Head: `Linear(1024 -> 1)`, dropout 0.1
- Loss: `SmoothL1Loss(beta=1.0)`
- Optimizer: AdamW, encoder LR `1.5e-5`, head LR `1e-3`, weight_decay `0.01`
- Schedule: 10% warmup + linear decay, bf16, grad clip 1.0
- 5 folds × 3 seeds [252, 253, 254] = 15 models
- `BATCH_TRAIN = 8`, `GRAD_ACCUM_STEPS = 2` (effective batch = 16)
- Per-fold best epoch by validation round-QWK
- Constrained threshold tuner (lambda=0.5)

**Time budget on A100:** longer context = ~1.5× slower per fold → **~50-65 min total** (vs 35-45 for step 12).

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q --upgrade "transformers>=4.41" "accelerate>=0.30" "sentencepiece>=0.2" scikit-learn pandas "numpy<2" scipy

## 2. Data setup (Colab / Kaggle / local)

Auto-detects platform and locates `asp_data*.zip`. Same logic as step 8.

Source priority:
1. `ASP_DATA_ZIP` env var
2. `asp_data*.zip` in cwd or any parent (up to repo root)
3. Kaggle: any `*.zip` under `/kaggle/input/`
4. Colab: interactive upload widget (fallback)

In [ ]:
import os, pathlib, zipfile, shutil

def _detect_platform():
    if 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_GPU' in os.environ:
        return 'colab'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or pathlib.Path('/kaggle/working').exists():
        return 'kaggle'
    return 'local'

PLATFORM = _detect_platform()
print('platform =', PLATFORM)

if PLATFORM == 'colab':
    WORK = pathlib.Path('/content/work')
elif PLATFORM == 'kaggle':
    WORK = pathlib.Path('/kaggle/working/asp_work')
else:
    WORK = pathlib.Path.cwd() / 'work'

DATA = WORK / 'data'
OUT = WORK / 'outputs'
RUN_DIR = OUT / 'e5_large_finetune_max384'
for d in [WORK, DATA, OUT, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def _find_local_zip():
    env_zip = os.environ.get('ASP_DATA_ZIP')
    if env_zip and pathlib.Path(env_zip).exists():
        return pathlib.Path(env_zip)
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        for pat in ('asp_data*.zip', 'asp_data.zip'):
            for cand in sorted(base.glob(pat)):
                return cand
        if (base / '.git').exists():
            break
    return None

def _find_kaggle_zip():
    root = pathlib.Path('/kaggle/input')
    if not root.exists():
        return None
    for cand in sorted(root.rglob('asp_data*.zip')):
        return cand
    for cand in sorted(root.rglob('*.zip')):
        return cand
    return None

def _ingest_zip(zip_path):
    print('using zip:', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA)
    print('extracted ->', DATA)

zip_path = _find_local_zip()
if zip_path is None and PLATFORM == 'kaggle':
    zip_path = _find_kaggle_zip()

if zip_path is not None:
    _ingest_zip(zip_path)
elif PLATFORM == 'colab':
    from google.colab import files
    print('No local zip found; falling back to Colab upload widget.')
    uploaded = files.upload()
    for name, content in uploaded.items():
        target = WORK / name
        target.write_bytes(content)
        if name.lower().endswith('.zip'):
            _ingest_zip(target)
else:
    raise FileNotFoundError(
        'No data zip found. Set ASP_DATA_ZIP=/path/to/asp_data.zip, place '
        'asp_data*.zip in cwd or a parent, or attach the dataset on Kaggle.')

print('\nFiles in DATA:')
for p in sorted(DATA.glob('*')):
    print(' ', p.name, p.stat().st_size)

## 3. Load + merge

In [ ]:
import pandas as pd, numpy as np, json, re, time
from pathlib import Path

DATA = Path(DATA)
RUN_DIR = Path(RUN_DIR)

train = pd.read_csv(DATA / 'train.csv')
public = pd.read_csv(DATA / 'public_test.csv')
private = pd.read_csv(DATA / 'private_test.csv')
sample = pd.read_csv(DATA / 'Test_Submission.csv')
abstracts_file = DATA / 'abstracts_merged_v2.csv'
if not abstracts_file.exists():
    abstracts_file = DATA / 'abstracts_merged_v3.csv'
abstracts = pd.read_csv(abstracts_file)
print('using abstract cache:', abstracts_file.name)

abs_map = abstracts[['source_split', 'id', 'abstract', 'has_abstract']]

def attach(df, split):
    df = df.copy()
    df['source_split'] = split
    out = df.merge(abs_map, on=['source_split', 'id'], how='left')
    out['abstract'] = out['abstract'].fillna('')
    out['has_abstract'] = out['has_abstract'].fillna(False).astype(bool)
    return out

train_full = attach(train, 'train').reset_index(drop=True)
public_full = attach(public, 'public_test').reset_index(drop=True)
private_full = attach(private, 'private_test').reset_index(drop=True)
for name, df in [('train', train_full), ('public', public_full), ('private', private_full)]:
    print(f'{name}: rows={len(df)}, has_abstract={int(df["has_abstract"].sum())} ({df["has_abstract"].mean():.1%})')

## 4. Tokenizer + dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'intfloat/e5-large-v2'
MAX_LEN = 384
BATCH_TRAIN = 8        # e5-large at bf16 on A100 40GB peaks ~22GB at this batch (BERT-large)
BATCH_EVAL = 16  # halved because max_len 384 (was 32 at max_len 256)
GRAD_ACCUM_STEPS = 2   # effective batch = 16 (matches step 6 SciNCL / step 9 BGE)

# E5 was trained with a mandatory 'query: ' or 'passage: ' prefix on every input.
# Each paper here is a single document we want to embed semantically -> 'passage: '.
# Without the prefix, the encoder's contrastive geometry is mis-aligned and OOF
# can drop by 0.05+. This is the ONE difference vs BGE step 9.
E5_PREFIX = 'passage: '

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
SEP = tokenizer.sep_token

def build_input(title, abstract):
    title = '' if pd.isna(title) else str(title).strip()
    abstract = '' if pd.isna(abstract) else str(abstract).strip()
    body = f'{title}{SEP}{abstract}' if abstract else title
    return f'{E5_PREFIX}{body}'

class PaperDataset(Dataset):
    def __init__(self, df, with_label):
        self.texts = [build_input(t, a) for t, a in zip(df['title'], df['abstract'])]
        self.labels = df['Label'].astype(np.float32).to_numpy() if with_label else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {'text': self.texts[idx], 'idx': idx}
        if self.labels is not None:
            item['label'] = self.labels[idx]
        return item

def collate(batch):
    texts = [b['text'] for b in batch]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN,
                    return_tensors='pt', return_token_type_ids=False)
    out = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'],
           'idx': torch.tensor([b['idx'] for b in batch], dtype=torch.long)}
    if 'label' in batch[0]:
        out['label'] = torch.tensor([b['label'] for b in batch], dtype=torch.float32)
    return out

print('tokenizer loaded:', MODEL_NAME, '; SEP =', SEP, '; prefix =', repr(E5_PREFIX))

## 5. Model

In [ ]:
import torch.nn as nn

class E5Regressor(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.1):
        super().__init__()
        # Force fp32 load (lesson L11): some HF checkpoints ship as fp16,
        # which crashes a freshly-built fp32 head with mat1/mat2 dtype mismatch.
        # transformers v5 renamed torch_dtype -> dtype.
        try:
            self.encoder = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        except TypeError:
            self.encoder = AutoModel.from_pretrained(model_name, dtype=torch.float32)
        self.encoder = self.encoder.float()
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.encoder.config.hidden_size, 1)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Mean pooling with attention mask.
        # E5 was trained with mean pooling (not CLS); using its native pool
        # preserves the contrastive-similarity geometry the encoder learned.
        # Same as BGE — both encoders need the masked mean over all tokens.
        last_hidden = outputs.last_hidden_state                           # (B, T, H)
        mask = attention_mask.unsqueeze(-1).float()                       # (B, T, 1)
        summed = (last_hidden * mask).sum(dim=1)                          # (B, H)
        counts = mask.sum(dim=1).clamp(min=1.0)                           # (B, 1)
        pooled = summed / counts                                          # (B, H)
        return self.head(self.dropout(pooled)).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

## 6. Single-fold training (bf16 autocast + grad accum)

In [ ]:
from sklearn.metrics import cohen_kappa_score
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS = 5
LR_ENCODER = 1.5e-5    # bge-large is bigger than BERT-base; 2e-5 too hot
LR_HEAD = 1e-3         # standard BERT-class head LR (bge is BERT-large arch, safe)
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

# bge-large is BERT-large architecture — no disentangled-attention overflow.
# bf16 is preferred (no GradScaler needed); fp16 fallback on older GPUs.
USE_BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

@torch.no_grad()
def predict(model, loader):
    model.eval()
    n = len(loader.dataset)
    out = np.zeros(n, dtype=np.float32)
    for batch in loader:
        ids = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            preds = model(ids, mask).float().cpu().numpy()
        idx = batch['idx'].numpy()
        out[idx] = preds
    if not np.isfinite(out).all():
        n_bad = int((~np.isfinite(out)).sum())
        print(f'  WARNING: predict() produced {n_bad} non-finite scores; clamping to 3.0')
        out = np.where(np.isfinite(out), out, 3.0)
    return np.clip(out, 1.0, 5.0)

def train_one_fold(train_df, valid_df, public_df, private_df, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_loader = DataLoader(PaperDataset(train_df, with_label=True),
                              batch_size=BATCH_TRAIN, shuffle=True,
                              collate_fn=collate, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(PaperDataset(valid_df, with_label=True),
                              batch_size=BATCH_EVAL, shuffle=False,
                              collate_fn=collate, num_workers=2, pin_memory=True)
    public_loader = DataLoader(PaperDataset(public_df.assign(Label=0), with_label=False),
                               batch_size=BATCH_EVAL, shuffle=False,
                               collate_fn=collate, num_workers=2, pin_memory=True)
    private_loader = DataLoader(PaperDataset(private_df.assign(Label=0), with_label=False),
                                batch_size=BATCH_EVAL, shuffle=False,
                                collate_fn=collate, num_workers=2, pin_memory=True)

    model = E5Regressor().to(device)
    no_decay = ['bias', 'LayerNorm.weight']
    enc_p = list(model.encoder.named_parameters())
    head_p = list(model.head.named_parameters())
    param_groups = [
        {'params': [p for n, p in enc_p if not any(nd in n for nd in no_decay)],
         'weight_decay': WEIGHT_DECAY, 'lr': LR_ENCODER},
        {'params': [p for n, p in enc_p if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0, 'lr': LR_ENCODER},
        {'params': [p for _, p in head_p], 'weight_decay': WEIGHT_DECAY, 'lr': LR_HEAD},
    ]
    optim = AdamW(param_groups)
    total_steps = (EPOCHS * len(train_loader)) // GRAD_ACCUM_STEPS
    scheduler = get_linear_schedule_with_warmup(
        optim, num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps)
    loss_fn = nn.SmoothL1Loss(beta=1.0)
    scaler = None if USE_BF16 else torch.amp.GradScaler('cuda')

    best_state = None
    best_qwk = -1.0
    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        n_skipped = 0
        t0 = time.time()
        optim.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            ids = batch['input_ids'].to(device, non_blocking=True)
            mask = batch['attention_mask'].to(device, non_blocking=True)
            y = batch['label'].to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                preds = model(ids, mask)
                loss = loss_fn(preds, y) / GRAD_ACCUM_STEPS
            if not torch.isfinite(loss):
                n_skipped += 1
                optim.zero_grad(set_to_none=True)
                continue
            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                if scaler is not None:
                    scaler.unscale_(optim)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                if scaler is not None:
                    scaler.step(optim)
                    scaler.update()
                else:
                    optim.step()
                scheduler.step()
                optim.zero_grad(set_to_none=True)
            running += loss.item() * GRAD_ACCUM_STEPS
        val_scores = predict(model, valid_loader)
        rounded = np.clip(np.round(val_scores), 1, 5).astype(int)
        qwk = cohen_kappa_score(valid_df['Label'].astype(int).to_numpy(), rounded, weights='quadratic')
        skipped_msg = f'  skipped={n_skipped}' if n_skipped else ''
        print(f'  epoch {epoch+1}/{EPOCHS}  loss={running/max(len(train_loader),1):.3f}  val_round_QWK={qwk:.4f}  ({time.time()-t0:.1f}s){skipped_msg}')
        if qwk > best_qwk:
            best_qwk = qwk
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is None:
        raise RuntimeError('Training produced no usable epoch (all NaN). Check LR / precision.')
    model.load_state_dict(best_state)
    return predict(model, valid_loader), predict(model, public_loader), predict(model, private_loader), best_qwk

print(f'train_one_fold ready (amp_dtype={"bf16" if USE_BF16 else "fp16"}, mean-pool, LR_enc={LR_ENCODER})')

## 7. Repeated CV (5 folds x 3 seeds)

In [ ]:
from sklearn.model_selection import StratifiedKFold

FOLDS = 5
SEEDS = [252, 253, 254]

y_class = train_full['Label'].astype(int).to_numpy()
oof_sum = np.zeros(len(train_full), dtype=np.float64)
oof_count = np.zeros(len(train_full), dtype=np.float64)
public_sum = np.zeros(len(public_full), dtype=np.float64)
private_sum = np.zeros(len(private_full), dtype=np.float64)
n_models = 0
fold_log = []

for seed in SEEDS:
    cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(train_full, y_class), start=1):
        print(f'\n=== seed={seed} fold={fold}/{FOLDS} ===')
        t0 = time.time()
        train_df = train_full.iloc[tr_idx].reset_index(drop=True)
        valid_df = train_full.iloc[va_idx].reset_index(drop=True)
        val_scores, pub_scores, priv_scores, best_qwk = train_one_fold(
            train_df, valid_df, public_full, private_full, seed * 1000 + fold)
        oof_sum[va_idx] += val_scores
        oof_count[va_idx] += 1.0
        public_sum += pub_scores
        private_sum += priv_scores
        n_models += 1
        fold_log.append({'seed': seed, 'fold': fold, 'best_round_qwk': float(best_qwk),
                         'minutes': round((time.time()-t0)/60, 2)})

print('\n=== Done. trained', n_models, 'models. ===')
oof_scores = oof_sum / np.clip(oof_count, 1.0, None)
public_scores = public_sum / n_models
private_scores = private_sum / n_models
print('OOF round-QWK:',
      cohen_kappa_score(y_class, np.clip(np.round(oof_scores), 1, 5).astype(int), weights='quadratic'))

## 8. Constrained threshold tuning

In [ ]:
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score, mean_absolute_error

TRAIN_DIST = pd.Series(y_class).value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0).to_numpy()
DIST_PENALTY_LAMBDA = 0.5

def scores_to_labels(scores, thresholds):
    return np.digitize(scores, np.sort(np.asarray(thresholds, dtype=float))) + 1

def predicted_dist(labels):
    return pd.Series(labels).value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0).to_numpy()

def tune_thresholds_constrained(y_true, oof, lambd=DIST_PENALTY_LAMBDA, seed=42):
    def objective(raw):
        thr = np.sort(raw)
        gap = np.min(np.diff(thr))
        gap_pen = 0.0 if gap >= 0.03 else (0.03 - gap) * 5.0
        labels = scores_to_labels(oof, thr)
        qwk = cohen_kappa_score(y_true, labels, weights='quadratic')
        dist_pen = float(np.sum(np.abs(predicted_dist(labels) - TRAIN_DIST)))
        return -qwk + gap_pen + lambd * dist_pen
    bounds = [(1.4, 2.5), (1.8, 2.9), (2.2, 3.4), (2.6, 4.2)]
    res = differential_evolution(objective, bounds, seed=seed, maxiter=120, popsize=15,
                                 polish=True, updating='immediate', workers=1)
    thr = np.sort(res.x)
    return thr, cohen_kappa_score(y_true, scores_to_labels(oof, thr), weights='quadratic')

thresholds, oof_qwk = tune_thresholds_constrained(y_class, oof_scores)
oof_pred = scores_to_labels(oof_scores, thresholds)
print('Constrained-tuned OOF QWK =', round(oof_qwk, 4))
print('  (anchors: E5_step12 0.6496, BGE 0.6404, SPECTER2 0.6373, SciNCL 0.6269, Ridge 0.5967)')
print('  (ablation hypothesis: max_len=384 should beat step12 by >= 0.005 OOF QWK)')
print('thresholds =', thresholds.tolist())
print('OOF predicted dist =', dict(zip([1,2,3,4,5], predicted_dist(oof_pred).round(3).tolist())))
print('TRAIN actual dist  =', dict(zip([1,2,3,4,5], TRAIN_DIST.round(3).tolist())))
print('OOF MAE =', round(mean_absolute_error(y_class, oof_pred), 4))
print('OOF macro-F1 =', round(f1_score(y_class, oof_pred, average='macro'), 4))

## 9. Save artefacts

In [ ]:
public_pred = scores_to_labels(public_scores, thresholds)
private_pred = scores_to_labels(private_scores, thresholds)

metrics = {
    'method': 'e5_large_finetune_max384',
    'model': MODEL_NAME,
    'folds': FOLDS,
    'seeds': SEEDS,
    'epochs': EPOCHS,
    'max_len': MAX_LEN,
    'batch_train': BATCH_TRAIN,
    'grad_accum_steps': GRAD_ACCUM_STEPS,
    'lr_encoder': LR_ENCODER,
    'lr_head': LR_HEAD,
    'pooling': 'mean',
    'amp_dtype': 'bf16' if USE_BF16 else 'fp16',
    'oof_qwk': float(oof_qwk),
    'oof_mae': float(mean_absolute_error(y_class, oof_pred)),
    'oof_macro_f1': float(f1_score(y_class, oof_pred, average='macro')),
    'thresholds': [float(v) for v in thresholds],
    'label_distribution_combined': {int(k): int(v) for k, v in pd.Series(
        np.concatenate([public_pred, private_pred])).value_counts().sort_index().items()},
    'label_distribution_public': {int(k): int(v) for k, v in pd.Series(public_pred).value_counts().sort_index().items()},
    'label_distribution_private': {int(k): int(v) for k, v in pd.Series(private_pred).value_counts().sort_index().items()},
    'fold_log': fold_log,
}
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

pd.DataFrame({'id': train_full['id'], 'Label': y_class,
              'oof_score': oof_scores, 'oof_pred': oof_pred}).to_csv(
    RUN_DIR / 'oof_scores.csv', index=False)
pd.DataFrame({'id': public_full['id'], 'score': public_scores,
              'pred': public_pred}).to_csv(RUN_DIR / 'public_scores.csv', index=False)
pd.DataFrame({'id': private_full['id'], 'score': private_scores,
              'pred': private_pred}).to_csv(RUN_DIR / 'private_scores.csv', index=False)

combo = pd.concat([
    pd.DataFrame({'id': public_full['id'], 'Label': public_pred}),
    pd.DataFrame({'id': private_full['id'], 'Label': private_pred}),
], ignore_index=True)
submission = sample[['id']].merge(combo, on='id', how='left')
submission['Label'] = submission['Label'].astype(int)
submission.to_csv(RUN_DIR / 'e5_large_finetune_max384_submission.csv', index=False)
print('submission rows =', len(submission))

## 10. Zip + download (Colab / Kaggle / local)

In [ ]:
zip_path = pathlib.Path(OUT) / 'e5_large_finetune_max384_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in pathlib.Path(RUN_DIR).iterdir():
        zf.write(p, arcname=f'e5_large_finetune_max384/{p.name}')
print('zipped:', zip_path, 'size MB =', round(zip_path.stat().st_size / 1e6, 2))

if PLATFORM == 'colab':
    from google.colab import files
    files.download(str(zip_path))
elif PLATFORM == 'kaggle':
    import shutil
    kaggle_out = pathlib.Path('/kaggle/working') / zip_path.name
    if zip_path.resolve() != kaggle_out.resolve():
        shutil.copy(zip_path, kaggle_out)
    print('available at:', kaggle_out)
else:
    print('local run — zip is at:', zip_path)